# 04 - Structured Streaming depuis Kafka (Partie 4, Personne 1)
Lecture du topic `trips`, parsing JSON, deux sinks : Parquet (MinIO) et KPI par fenêtre (console, cœur de la Personne 2).

In [1]:
import sys, time, subprocess
sys.path.append('/home/jovyan/work/pipeline')
from pedali_spark import get_spark
import kafka_stream as ks

spark = get_spark('pedali-04-streaming')
spark.sparkContext.setLogLevel('WARN')
STAMP = str(int(time.time()))
OUT = 's3a://lake/stream/trips_parsed_' + STAMP
CKPT = 's3a://lake/checkpoints/stream04_' + STAMP

In [2]:
stream = ks.read_trips_stream(spark, starting_offsets='earliest', max_offsets=1500)
stream.printSchema()
q1 = ks.parsed_sink(stream, OUT, CKPT + '/parquet')
q2 = ks.kpi_console_sink(spark, stream, CKPT + '/kpi')
print([q.name for q in spark.streams.active])

root
 |-- cle: string (nullable = true)
 |-- value: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_ts: timestamp (nullable = true)
 |-- t: struct (nullable = true)
 |    |-- trip_id: string (nullable = true)
 |    |-- user_id: string (nullable = true)
 |    |-- bike_id: string (nullable = true)
 |    |-- start_station_id: string (nullable = true)
 |    |-- end_station_id: string (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- start_time: timestamp (nullable = true)
 |    |-- end_time: timestamp (nullable = true)
 |    |-- duration_min: integer (nullable = true)
 |    |-- price_mad: double (nullable = true)
 |    |-- status: string (nullable = true)



['kpi_console', 'parsed_parquet']


## Production en continu pendant que les requêtes tournent
On ajoute 1 000 messages depuis le producteur Kafka (les 3 000 premiers sont déjà dans le topic).

In [3]:
print(subprocess.run(['python','/home/jovyan/work/producer/producer.py','--bootstrap','kafka:9092','--skip','3000','--limit','1000','--rate','400'],capture_output=True,text=True).stdout)
q1.processAllAvailable()
for q in (q1, q2): print(q.name, '->', q.lastProgress['numInputRows'] if q.lastProgress else None, 'lignes (dernier batch)')

1000 messages envoyes vers kafka:9092/trips



parsed_parquet -> 1003 lignes (dernier batch)
kpi_console -> None lignes (dernier batch)


In [4]:
back = spark.read.parquet(OUT)
print('Messages JSON valides ecrits en Parquet :', back.count())
back.groupBy('partition').count().orderBy('partition').show()
back.select('trip_id','city','status','price_mad','partition','offset').show(5)

Messages JSON valides ecrits en Parquet : 4000


+---------+-----+
|partition|count|
+---------+-----+
|        0| 1423|
|        1| 1687|
|        2|  890|
+---------+-----+



+--------+----------+---------+---------+---------+------+
| trip_id|      city|   status|price_mad|partition|offset|
+--------+----------+---------+---------+---------+------+
|T0017456|Casablanca|completed|     22.5|        1|     0|
|T0007752|     Rabat|completed|     10.5|        1|     1|
|T0013441|     Rabat|completed|      9.0|        1|     2|
|T0018856|     Rabat|completed|     27.0|        1|     3|
|T0001884|     Rabat|completed|     18.0|        1|     4|
+--------+----------+---------+---------+---------+------+
only showing top 5 rows



In [5]:
for q in spark.streams.active: q.stop()
spark.stop()